# 1 · Build dataset + re-train the 3 heads (the long GPU job)

Runs `tools/unified_retrain.py` end-to-end on ONE shared dataset + config (fp16), LLM frozen:
1. generate one answer/question (Instruct fp16) and cache SEP (135168-d) + HalluShift (71-d) features + BLEURT label,
2. re-fit the SEP probe + HalluShift MLP on those features (CPU),
3. re-train the TSV steering vector + centroids (base fp16),
4. assemble scores, train+CV the fusion, print per-detector vs fused AUROC/AUPR.

Why re-train: the *frozen* artifacts each only work on their own dataset/dtype and don't co-align on one shared generation; re-fitting the tiny heads on one config makes all three in-distribution together (the LLM stays frozen). Output → `data/<dataset>_fused.parquet` + re-trained artifacts.

**This is the long GPU pass** (two 8B model loads, ~minutes/question with attentions).

In [ ]:
import os, sys
os.environ.setdefault('HF_HOME', r'D:/LLAMA CACHE/huggingface')  # reuse local LLaMA cache
sys.path.insert(0, os.path.abspath(os.path.join('..', 'hallking')))
import warnings; warnings.filterwarnings('ignore')
DATASET='triviaqa'; N=1200; OFFSET=1000; EPOCHS_TSV=40
cmd=[sys.executable, os.path.join('..','tools','unified_retrain.py'),
     '--dataset',DATASET,'--n',str(N),'--offset',str(OFFSET),'--epochs_tsv',str(EPOCHS_TSV)]
import subprocess
_env = {**os.environ, 'PYTHONUNBUFFERED': '1'}
print('running:', ' '.join(cmd), flush=True)
_p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                      text=True, bufsize=1, env=_env)
for _line in _p.stdout:
    print(_line, end='', flush=True)
if _p.wait() != 0:
    raise RuntimeError(f'subprocess failed (rc={_p.returncode})')


In [ ]:
import pandas as pd
df = pd.read_parquet(os.path.join('..','data',f'{DATASET}_fused.parquet'))
print(df.shape, df['hallucination'].value_counts().to_dict())
df[['question','answer','sep_entropy','hallushift','tsv_margin','bleurt','hallucination']].head(8)